# 01 · Dummy's first call – models and the Responses API

In the garage, Tony's first helper was a clumsy robot arm called **Dummy**. It could do exactly one thing at a time. That is where we start too: **one model, one question, one answer**.

You will see:
1. the **Responses API** (the API agents are built on) and the older **Chat Completions** API
2. **streaming**, **instructions** and **structured output**
3. what a response object really contains (tokens, output items)
4. **embeddings**

Deep dive: [docs/models.md](../docs/models.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

project_openai = openai_client()      # OpenAI client bound to the Foundry project
CHAT_MODEL = model("chat")            # the deployment name from STARK_CHAT_MODEL
print("model deployment:", CHAT_MODEL)

## 1 · The Responses API
`model` is the **deployment name**. `input` can be a string or a list of messages. `show_output` ([stark_helpers.py](stark_helpers.py)) prints the response id, the model and the text.

**Expected:** one sentence: Dummy is the robot arm in Tony Stark's workshop.

In [ ]:
dummy_question = "In one sentence: what is Dummy in Iron Man?"
first_response = project_openai.responses.create(model=CHAT_MODEL, input=dummy_question)
show_output(first_response)

### Instructions (the system prompt) and limits
`instructions` steers behaviour for this call only. `max_output_tokens` caps cost.

**Goal:** one change from the call above. The same model gets a role and a word limit.

**Expected:** an answer in Dummy's voice, at most 12 words, ending with a beep.

> 💡 `max_output_tokens` also counts reasoning tokens. If a reasoning model comes back with an empty answer, the cap was too low.

In [ ]:
DUMMY_INSTRUCTIONS = "You are Dummy, a clumsy robot arm. Answer in at most 12 words and end with a beep."
smoothie_question = "Did you spill the smoothie again?"
dummy_response = project_openai.responses.create(
    model=CHAT_MODEL,
    instructions=DUMMY_INSTRUCTIONS,
    input=smoothie_question,
    max_output_tokens=200,
)
show_output(dummy_response)

## 2 · Chat Completions (the classic shape)
The same deployment also speaks Chat Completions. New work should use Responses: it is stateful, supports tools and agents. Chat Completions is still handy for existing code.

**Expected:** a short list of AIs. Which ones the model names varies ❓; the point is the shape of the call: `messages` in, `choices[0].message.content` out.

In [ ]:
ai_question = "Name the three MCU AIs that came before Vision. Short list."
completion = project_openai.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": ai_question}],
)
print(completion.choices[0].message.content)

## 3 · Streaming
Events arrive as the model writes. `response.output_text.delta` carries the text pieces.

**Expected:** a 3-line status report that appears piece by piece, then `[done] tokens: <total>`.

In [ ]:
fire_drill_prompt = "Write a 3-line status report of a workshop fire drill."
stream = project_openai.responses.create(model=CHAT_MODEL, input=fire_drill_prompt, stream=True)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print("\n\n[done] tokens:", event.response.usage.total_tokens)

## 4 · Structured output
Ask for JSON that matches a schema. `strict` makes the model follow it exactly, so your code can parse the answer without guessing.

**Expected:** a dict with exactly the keys `suit`, `mark` and `weaknesses`, parsed by `json.loads` without any cleanup. The schema fixes the keys and types; the values come from the model (`mark` should be `3`).

In [ ]:
import json
suit_card_schema = {
    "type": "object",
    "properties": {
        "suit": {"type": "string"},
        "mark": {"type": "integer"},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["suit", "mark", "weaknesses"],
    "additionalProperties": False,
}
suit_question = "Describe the Mark 3 armor."
suit_response = project_openai.responses.create(
    model=CHAT_MODEL,
    input=suit_question,
    text={"format": {"type": "json_schema", "name": "suit_card", "schema": suit_card_schema, "strict": True}},
)
suit_card = json.loads(suit_response.output_text)
print(json.dumps(suit_card, indent=2))

## 5 · What is inside a response?
A response is more than text: `output` is a list of **items** (messages, tool calls, reasoning), `usage` counts tokens (this is what you pay for). This cell looks at `suit_response` from section 4, so run that first.

**Expected:** status `completed`, the token counts, and the item types. With a reasoning model as the chat deployment, a `reasoning` item comes before the `message` ❓.

In [ ]:
print("status :", suit_response.status)
print("usage  :", suit_response.usage.input_tokens, "in /", suit_response.usage.output_tokens, "out /", suit_response.usage.total_tokens, "total")
for item in suit_response.output:
    print("item   :", item.type)

## 6 · Embeddings
Embeddings turn text into vectors for search and retrieval. This uses the embedding deployment, not the chat model. Cosine similarity compares two vectors: close to 1 means close in meaning.

**Expected:** `3 vectors of 1536 dimensions` (with the default `text-embedding-3-small`), and a higher similarity for *arc reactor ~ repulsor* than for *arc reactor ~ smoothie*.

In [ ]:
import math

words = ["arc reactor", "repulsor", "smoothie"]
embedding_result = embeddings_client().embeddings.create(model=model("embedding"), input=words)
vectors = [d.embedding for d in embedding_result.data]
print(len(vectors), "vectors of", len(vectors[0]), "dimensions")


def cosine_similarity(u, v):
    """Cosine of the angle between two vectors: 1 = same direction, 0 = unrelated."""
    return sum(x * y for x, y in zip(u, v)) / (math.sqrt(sum(x * x for x in u)) * math.sqrt(sum(y * y for y in v)))


print("arc reactor ~ repulsor :", round(cosine_similarity(vectors[0], vectors[1]), 3))
print("arc reactor ~ smoothie :", round(cosine_similarity(vectors[0], vectors[2]), 3))

## 👀 In the portal
**Build → Models →** your chat deployment **→ Open in playground**: the same model, with the same *instructions* field, and no code. Handy for trying a prompt before you write it down.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up
Nothing to delete by hand. Responses are stored by default, though: chapter 02 shows what that means and how to delete them.

## 🎯 Recap
| Need | Use |
|---|---|
| One-off answer, new code | `responses.create(model=..., input=...)` |
| Existing Chat Completions code | `chat.completions.create(...)` |
| Parseable output | `text={"format": {"type": "json_schema", ...}}` |
| Search / similarity | `embeddings.create(...)` |

Dummy has no memory: every call starts from zero.

> 💡 Next: [02 · Conversations](02_conversations.ipynb)